<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day05-discussion-2.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 5 — In-class discussion problem (2 of 3)

Work this out **by hand in your group first** — then run the code cell to check your answer before presenting.

## Does the pseudocount SCHEME change which sequence wins?

The book page adds a uniform pseudocount of +1 to every residue at every position. A commonly used alternative instead makes the pseudocount *proportional to each residue's own background frequency* — a rare residue like tryptophan gets a smaller pseudocount than a common one like alanine, rather than the same +1 for both.

Using the same 5-sequence toy alignment as the book page:

```
Position:   1 2 3 4 5
Protein 1:  A G L S P
Protein 2:  A G L T P
Protein 3:  R G I S P
Protein 4:  A A L S Q
Protein 5:  A A L S P
```

1. Predict: will background-weighted pseudocounts change *which* sequence scores highest, compared to uniform pseudocounts?
2. Predict whether the position-1 score for `G` (which never appears at position 1) will be more negative, less negative, or the same under background-weighting compared to uniform +1.
3. Run the cell below to check both predictions.

In [1]:
import math

msa = ["AGLSP", "AGLTP", "RGISP", "AALSQ", "AALSP"]
background = {
    "A": 0.074, "R": 0.042, "G": 0.074, "L": 0.076, "S": 0.081,
    "T": 0.062, "P": 0.050, "Q": 0.037, "I": 0.038,
}
residues = sorted(background.keys())
n_pos = len(msa[0])

def counts_at(pos):
    col = [seq[pos] for seq in msa]
    return {r: col.count(r) for r in residues}

def best_sequence(matrix):
    best = [max(col, key=col.get) for col in matrix]
    score = sum(matrix[i][best[i]] for i in range(len(best)))
    return "".join(best), score

def pssm_uniform(pseudocount=1):
    matrix = []
    for pos in range(n_pos):
        c = counts_at(pos)
        total = sum(c.values()) + pseudocount * len(residues)
        freqs = {r: (c[r] + pseudocount) / total for r in residues}
        matrix.append({r: math.log2(freqs[r] / background[r]) for r in residues})
    return matrix

def pssm_bgweighted(total_pseudo=9):
    matrix = []
    for pos in range(n_pos):
        c = counts_at(pos)
        n_seq = sum(c.values())
        pseudo = {r: total_pseudo * background[r] for r in residues}
        total = n_seq + sum(pseudo.values())
        freqs = {r: (c[r] + pseudo[r]) / total for r in residues}
        matrix.append({r: math.log2(freqs[r] / background[r]) for r in residues})
    return matrix

m_uniform = pssm_uniform()
m_weighted = pssm_bgweighted()

print("Winner, uniform pseudocounts:      ", best_sequence(m_uniform))
print("Winner, background-weighted:       ", best_sequence(m_weighted))
print()
print("Position 1, G score, uniform pseudocount:      ", round(m_uniform[0]["G"], 2))
print("Position 1, G score, background-weighted:      ", round(m_weighted[0]["G"], 2))

Winner, uniform pseudocounts:       ('AGLSP', 11.429318755211028)
Winner, background-weighted:        ('AGLSP', 13.429531824745402)

Position 1, G score, uniform pseudocount:       -0.05
Position 1, G score, background-weighted:       -0.12


**Discussion point:** both schemes agree on the winning sequence (`AGLSP`, the alignment's own consensus) — the *ranking* of residues at each position doesn't change, since both schemes still favor whatever's actually observed. What changes is *how confidently* the PSSM makes that call: background-weighting gives glycine (background frequency 7.4%, roughly average) a slightly smaller pseudocount than a fixed +1 would, so its position-1 score (where it never appears) ends up **more negative**, not less — the model is more willing to penalize a residue's absence once its pseudocount reflects how "expected" it would have been anyway. Neither scheme is simply "more correct" — they encode different assumptions about what a zero count means.